In [1]:
import pandas as pd
import numpy as np
from torchvision.models import resnet18
from torchvision.transforms import v2
import torch
from torch import Tensor
from torch.utils.data import Dataset, DataLoader
from typing import Optional, Callable, Union
from tqdm import tqdm
import pydicom
from PIL import Image

In [2]:
# set constants
IMAGE_SIZE: int = 224
BATCH_SIZE: int = 256
MODEL_WEIGHTS_PATH: str = "optimized_resnet18_implant_class.pth"
DATA_PATH: str = "existing-code/cnn_data/resnet18_test_data.csv"

In [3]:
class PandasDataset(Dataset):
    """
    Expects images to be saved as PNGs
    """
    def __init__(
        self,
        df: pd.DataFrame,
        img_col: str,
        transforms: Optional[Callable[[any], Tensor]] = None,
    ) -> None:
        self.df = df
        self.transforms: Optional[Callable[[any], Tensor]] = transforms

        # extract the list of image paths
        self._img_path_list = self.df[img_col].tolist()

    def _load_img(self, idx: int) -> tuple[Image, str]:
        path = self._img_path_list[idx]
        
        tensor = load_dicom_as_tensor(path)
        return tensor, path

    def to_loader(self, batch_size: int, **kwargs) -> DataLoader:
        # cast the current dataset to a dataloader
        return DataLoader(self, batch_size=batch_size, **kwargs)

    def __len__(self) -> int:
        return len(self._img_path_list)

    def __getitem__(self, idx: int) -> Union[Tensor, tuple[Tensor, int]]:
        img, path = self._load_img(idx)
        
        # load the image and apply any desired transformations
        if self.transforms is not None:
            img: Tensor = self.transforms(img)
            
        return img, path


def load_dicom_as_tensor(dicom_path) -> Image:
    dicom = pydicom.dcmread(dicom_path)

    # Apply VOI LUT if present
    img = pydicom.pixels.apply_voi_lut(dicom.pixel_array, dicom)

    # Normalize pixel values to 0–255 and convert to uint8
    img = img.astype(np.float32)
    img = (img - np.min(img)) / (np.max(img) - np.min(img) + 1e-5)
    img = (img * 255).astype(np.uint8)

    img = np.repeat(img[..., np.newaxis], 3, axis=-1)

    # Convert to PIL image in RGB
    # img = Image.fromarray(img).convert("RGB")
    return img


## Load Model + Weights

In [4]:
GPU_NUM: int = 3

device = f"cuda:{GPU_NUM}" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# load the resnet18 model and its weights
model = resnet18(num_classes=2, weights=None)
model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, weights_only=True))
model.eval()
model.to(device)

Using device: cuda:3


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## Prepare Data

In [5]:
# define transforms
transforms = v2.Compose([
    v2.ToImage(), 
    v2.ToDtype(torch.float32, scale=True),
    v2.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    v2.Normalize([0.5]*3, [0.5]*3)
])
# load the inference dataframe
inference_df = pd.read_csv(DATA_PATH)

# load the dataframe into a dataset and dataloader
inference_dataset: PandasDataset = PandasDataset(inference_df, img_col="anon_dicom_path", transforms=transforms)
inference_dataloader: DataLoader = inference_dataset.to_loader(batch_size=BATCH_SIZE, shuffle=False)

## Run Inference

In [6]:
path_list: list[str] = []
pred_list: list[float] = []

# iterate over the batches in the dataloader
for batch_images, batch_paths in tqdm(inference_dataloader):
    # cast images to the device
    batch_images: Tensor = batch_images.to(device)

    # run inference on the batch
    batch_preds: Tensor = model(batch_images)

    # apply softmax, argmax, then detach the tensor and cast to a list
    batch_preds: list[np.int64] = list(
        torch.nn.functional.softmax(batch_preds, dim=1)
            .argmax(axis=1)
            .detach()
            .cpu()
            .numpy()
    )

    path_list.extend(list(batch_paths))
    pred_list.extend(batch_preds)

100%|██████████| 5/5 [10:19<00:00, 123.96s/it]


## Map Predictions to the Data

In [7]:
# zip the image paths and corresonding predictions into a dict
preds_dict: dict[str, np.int64] = dict(zip(path_list, pred_list))

inference_df["prediction"] = inference_df["anon_dicom_path"].map(preds_dict)
inference_df["prediction"].value_counts(dropna=False)

prediction
0    646
1    604
Name: count, dtype: int64

In [14]:
inference_df["tp"] = ((inference_df["implant_visible"] == 1) & (inference_df["prediction"] == 1)).astype(int)
inference_df["tn"] = ((inference_df["implant_visible"] == 0) & (inference_df["prediction"] == 0)).astype(int)
inference_df["fp"] = ((inference_df["implant_visible"] == 0) & (inference_df["prediction"] == 1)).astype(int)
inference_df["fn"] = ((inference_df["implant_visible"] == 1) & (inference_df["prediction"] == 0)).astype(int)

tp = inference_df["tp"].sum()
tn = inference_df["tn"].sum()
fp = inference_df["fp"].sum()
fn = inference_df["fn"].sum()

accuracy: float = (tp + tn) / (tp + tn + fp + fn)
sensitivity: float = tp / (tp + fn)
specificity: float = tn / (tn + fp)

print(f"Test Accuracy: {accuracy:.2f}")
print(f"Test Sensitivity: {sensitivity:.2f}")
print(f"Test Specificity: {specificity:.2f}")

Test Accuracy: 0.98
Test Sensitivity: 0.97
Test Specificity: 1.00
